In [1]:
%pip install pygame
%pip install Pillow
%pip install requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import tkinter as tk  # 修正這一行
from tkinter import messagebox
from PIL import Image, ImageTk
import random
import pygame
import time
import os
import sys

# ... 後面代碼保持不變 ...

# ==========================================
#          📦 資源路徑處理 (For PyInstaller)
# ==========================================
def resource_path(relative_path):
    """ 獲取資源絕對路徑，兼容開發環境與 PyInstaller 打包環境 """
    try:
        base_path = sys._MEIPASS
    except Exception:
        base_path = os.path.abspath(".")
    return os.path.join(base_path, relative_path)

# ==========================================
#          📦 遊戲自定義設置 (Customization)
# ==========================================
WORD_BANK = ['IMPERIUM', 'HERESY', 'LOYALTY', 'SACRIFICE', 'CODEX', 'SERVITOR', 
             'ASTARTES', 'INQUISITOR','CUSTODES','MILITARUM','LOGISTICS',
             'BOLTER','TECHNOLOGY','WAR','CRUELTY','GRIM']
VICTIM_NAME = "Averyl Jingwei"
SPRITE_FILE = resource_path("boltgun.png") 

SOUNDS = {
    "correct": resource_path("correct.wav"),
    "load": resource_path("load_bolt.wav"),
    "bolt": resource_path("bolt_rack.wav"),
    "aim": resource_path("servo_aim.wav"),
    "safety": resource_path("click.wav"),
    "eat_bolt": resource_path("eat_boltgun.wav"),
    "fire": resource_path("bolter_shot.wav"),
    "eject": resource_path("shell_eject.wav")
}

STAGES = [
    "1. 處刑人取出了一枚 .75 口徑爆彈...",
    "2. 彈藥壓入彈匣。機魂正在甦醒。",
    "3. 槍機上膛。金屬碰撞聲在審訊室迴盪。",
    "4. 爆彈槍已對準罪人的頭顱。",
    "5. 保險關閉。手指已扣在扳機上。",
    "6. [ 已淨化 ]"
]

NARRATIVES = [
    "知識的代價是血。只有正確的供詞能救你。",
    "帝皇保佑那些言辭準確的人。再試一次。",
    "一個錯字，一個靈魂的隕落。聽聽那機魂的咆哮。",
    "沈默並不能保護你，只有真理可以。",
    "你是忠誠的僕人，還是隱藏的異端？"
]

class BolterTrial:
    def __init__(self, root):
        self.root = root
        self.root.title("審判庭：爆彈槍審訊系統")
        self.root.geometry("600x850")
        self.root.configure(bg="#050505")

        pygame.mixer.init()

        self.word = random.choice(WORD_BANK).upper()
        self.guessed_letters = set()
        self.mistakes = 0
        self.max_mistakes = len(STAGES) - 1

        try:
            img = Image.open(SPRITE_FILE)
            img = img.resize((250, 150), Image.Resampling.LANCZOS)
            self.boltgun_sprite = ImageTk.PhotoImage(img)
        except Exception as e:
            self.boltgun_sprite = None

        self.canvas = tk.Canvas(root, width=500, height=350, bg="#111", highlightthickness=2, highlightbackground="#444")
        self.canvas.pack(pady=20)
        
        self.status_label = tk.Label(root, text="--- 審訊開始：請輸入證詞 ---", fg="#ffcc00", bg="#050505", font=("Microsoft JhengHei", 12, "bold"))
        self.status_label.pack()

        self.word_display = tk.Label(root, text=self.get_display_word(), fg="#fff", bg="#050505", font=("Courier", 40, "bold"))
        self.word_display.pack(pady=20)

        self.narrative_display = tk.Label(root, text=NARRATIVES[0], fg="#888", bg="#050505", font=("Microsoft JhengHei", 10, "italic"), wraplength=500)
        self.narrative_display.pack(pady=10)

        self.entry = tk.Entry(root, font=("Courier", 22), width=5, justify='center', bg="#1a1a1a", fg="#ffcc00", insertbackground="white")
        self.entry.pack()
        self.entry.focus_set()
        self.entry.bind('<Return>', lambda e: self.make_guess())

        self.draw_trial()

    def screen_shake(self, intensity=20, duration=500):
        start_time = time.time() * 1000
        orig_x = self.root.winfo_x()
        orig_y = self.root.winfo_y()

        def shake():
            if (time.time() * 1000) - start_time < duration:
                dx = random.randint(-intensity, intensity)
                dy = random.randint(-intensity, intensity)
                self.root.geometry(f"+{orig_x + dx}+{orig_y + dy}")
                self.root.after(20, shake)
            else:
                self.root.geometry(f"+{orig_x}+{orig_y}")
        shake()

    def draw_explosion(self):
        """ 在受刑人頭部位置繪製紅橘色閃爆效果 """
        colors = ["#FF4500", "#FF8C00", "#FFD700", "#FFFFFF"] # 紅, 橘, 金, 白
        for _ in range(15):
            x = random.randint(310, 390)
            y = random.randint(90, 170)
            size = random.randint(10, 40)
            color = random.choice(colors)
            self.canvas.create_oval(x-size, y-size, x+size, y+size, fill=color, outline="")

    def play_sfx(self, sound_key):
        try:
            pygame.mixer.Sound(SOUNDS[sound_key]).play()
        except: pass

    def get_display_word(self):
        return " ".join([char if char in self.guessed_letters else "_" for char in self.word])

    def draw_trial(self):
        self.canvas.delete("all")
        self.canvas.create_oval(320, 100, 380, 160, outline="white", width=2) # 頭部
        self.canvas.create_line(350, 160, 350, 280, fill="white", width=2)    # 身體
        
        if self.mistakes >= 1:
            y_pos = 180 if self.mistakes < 4 else 140
            if self.boltgun_sprite:
                self.canvas.create_image(150, y_pos, image=self.boltgun_sprite)
            else:
                self.canvas.create_rectangle(50, y_pos-30, 250, y_pos+30, fill="#600", outline="#f00")
            
        if self.mistakes >= 4:
            self.canvas.create_line(230, 145, 320, 130, fill="red", width=2, dash=(5, 2))

    def make_guess(self):
        guess = self.entry.get().upper().strip()
        self.entry.delete(0, tk.END)
        if len(guess) != 1 or not guess.isalpha() or guess in self.guessed_letters: return

        self.guessed_letters.add(guess)
        if guess in self.word:
            self.play_sfx("correct")
            self.word_display.config(text=self.get_display_word())
            if "_" not in self.get_display_word(): self.end_game(True)
        else:
            self.mistakes += 1
            self.trigger_mistake_sequence()

    def trigger_mistake_sequence(self):
        self.status_label.config(text=STAGES[self.mistakes-1])
        self.narrative_display.config(text=random.choice(NARRATIVES))
        self.draw_trial()

        if self.mistakes < self.max_mistakes:
            s_map = {1: "load", 2: "load", 3: "bolt", 4: "aim", 5: "safety"}
            if self.mistakes in s_map: self.play_sfx(s_map[self.mistakes])
        else:
            self.play_sfx("eat_bolt")
            self.root.after(600, self.execute_fire)

    def execute_fire(self):
        self.play_sfx("fire")
        self.screen_shake(intensity=25)
        self.draw_explosion() # 繪製爆炸
        self.root.after(600, lambda: self.play_sfx("eject"))
        self.root.after(1500, lambda: self.end_game(False))

    def end_game(self, won):
        if won:
            messagebox.showinfo("審判結束", f"真理已明。{VICTIM_NAME} 被判無罪釋放。")
        else:
            msg = (f"{VICTIM_NAME} 在一場由拼寫錯誤引發的危機中喪生。\n"
                   f"[ 已淨化 ]\n"
                   f"我們將永遠記住知識、進步與錯誤的代價。\n"
                   f"正確的單字是: {self.word}")
            messagebox.showerror("帝皇的裁決", msg)
        self.root.destroy()

if __name__ == "__main__":
    root = tk.Tk()
    game = BolterTrial(root)
    root.mainloop()

c:\Users\kingo\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
import tkinter as tk
from tkinter import messagebox
from PIL import Image, ImageTk
import random
import pygame
import time
import os
import sys
import requests  # 新增：用於網路請求
from bs4 import BeautifulSoup  # 新增：用於解析網頁內容

# ==========================================
#          📦 資源路徑處理
# ==========================================
def resource_path(relative_path):
    try:
        base_path = sys._MEIPASS
    except Exception:
        base_path = os.path.abspath(".")
    return os.path.join(base_path, relative_path)

# ==========================================
#          🌐 維基百科爬蟲模組 (New!)
# ==========================================
def fetch_wiki_military_words():
    """ 從維基百科抓取軍事相關單字 """
    print("正在從維基百科獲取最新情報 (抓取單字)...")
    
    # 鎖定維基百科軍事術語或分類頁面
    urls = [
        "https://en.wikipedia.org/wiki/Glossary_of_military_terms",
        "https://en.wikipedia.org/wiki/Category:Military_units_and_formations"
    ]
    
    word_pool = []
    
    try:
        for url in urls:
            response = requests.get(url, timeout=5)
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # 抓取所有連結中的文字（通常是術語標題）
            for link in soup.find_all('a'):
                text = link.get_text().strip()
                # 過濾條件：純英文、長度 4-10、排除含有空格的片語
                if text.isalpha() and 4 <= len(text) <= 10:
                    word_pool.append(text.upper())
        
        # 如果抓取成功，回傳去重後的列表，否則使用備用清單
        if len(word_pool) > 10:
            print(f"情報獲取成功！已新增 {len(set(word_pool))} 個新術語。")
            return list(set(word_pool))
    except Exception as e:
        print(f"連線失敗，使用內建資料庫。原因: {e}")
    
    # 備用清單 (防止沒網路時當機)
    return ['IMPERIUM', 'HERESY', 'LOYALTY', 'SACRIFICE', 'CODEX', 'BOLTER', 'TACTICS', 'INFANTRY']

# ==========================================
#          📦 遊戲自定義設置
# ==========================================
# 初始載入：優先嘗試從網路獲取單字
WORD_BANK = fetch_wiki_military_words()

VICTIM_NAME = "Averyl Jingwei"
SPRITE_FILE = resource_path("boltgun.png")

SOUNDS = {
    "correct": resource_path("correct.wav"),
    "load": resource_path("load_bolt.wav"),
    "bolt": resource_path("bolt_rack.wav"),
    "aim": resource_path("servo_aim.wav"),
    "safety": resource_path("click.wav"),
    "eat_bolt": resource_path("eat_boltgun.wav"),
    "fire": resource_path("bolter_shot.wav"),
    "eject": resource_path("shell_eject.wav")
}

STAGES = [
    "1. 處刑人取出了一枚 .75 口徑爆彈...",
    "2. 彈藥壓入彈匣。機魂正在甦醒。",
    "3. 槍機上膛。金屬碰撞聲在審訊室迴盪。",
    "4. 爆彈槍已對準罪人的頭顱。",
    "5. 保險關閉。手指已扣在扳機上。",
    "6. [ 已淨化 ]"
]

NARRATIVES = [
    "知識的代價是血。只有正確的供詞能救你。",
    "帝皇保佑那些言辭準確的人。再試一次。",
    "一個錯字，一個靈魂的隕落。聽聽那機魂的咆哮。",
    "沈默並不能保護你，只有真理可以。",
    "你是忠誠的僕人，還是隱藏的異端？"
]

class BolterTrial:
    def __init__(self, root):
        self.root = root
        self.root.title("審判庭：爆彈槍審訊系統 (Wiki 聯網版)")
        self.root.geometry("600x850")
        self.root.configure(bg="#050505")

        pygame.mixer.init()

        self.word = random.choice(WORD_BANK).upper()
        self.guessed_letters = set()
        self.mistakes = 0
        self.max_mistakes = len(STAGES) - 1

        try:
            img = Image.open(SPRITE_FILE)
            img = img.resize((250, 150), Image.Resampling.LANCZOS)
            self.boltgun_sprite = ImageTk.PhotoImage(img)
        except Exception as e:
            self.boltgun_sprite = None

        self.canvas = tk.Canvas(root, width=500, height=350, bg="#111", highlightthickness=2, highlightbackground="#444")
        self.canvas.pack(pady=20)
       
        self.status_label = tk.Label(root, text="--- 審訊開始：請輸入證詞 ---", fg="#ffcc00", bg="#050505", font=("Microsoft JhengHei", 12, "bold"))
        self.status_label.pack()

        self.word_display = tk.Label(root, text=self.get_display_word(), fg="#fff", bg="#050505", font=("Courier", 40, "bold"))
        self.word_display.pack(pady=20)

        self.narrative_display = tk.Label(root, text=NARRATIVES[0], fg="#888", bg="#050505", font=("Microsoft JhengHei", 10, "italic"), wraplength=500)
        self.narrative_display.pack(pady=10)

        self.entry = tk.Entry(root, font=("Courier", 22), width=5, justify='center', bg="#1a1a1a", fg="#ffcc00", insertbackground="white")
        self.entry.pack()
        self.entry.focus_set()
        self.entry.bind('<Return>', lambda e: self.make_guess())

        self.draw_trial()

    def screen_shake(self, intensity=20, duration=500):
        start_time = time.time() * 1000
        orig_x = self.root.winfo_x()
        orig_y = self.root.winfo_y()

        def shake():
            if (time.time() * 1000) - start_time < duration:
                dx = random.randint(-intensity, intensity)
                dy = random.randint(-intensity, intensity)
                self.root.geometry(f"+{orig_x + dx}+{orig_y + dy}")
                self.root.after(20, shake)
            else:
                self.root.geometry(f"+{orig_x}+{orig_y}")
        shake()

    def draw_explosion(self):
        colors = ["#FF4500", "#FF8C00", "#FFD700", "#FFFFFF"]
        for _ in range(15):
            x = random.randint(310, 390)
            y = random.randint(90, 170)
            size = random.randint(10, 40)
            color = random.choice(colors)
            self.canvas.create_oval(x-size, y-size, x+size, y+size, fill=color, outline="")

    def play_sfx(self, sound_key):
        try:
            pygame.mixer.Sound(SOUNDS[sound_key]).play()
        except: pass

    def get_display_word(self):
        return " ".join([char if char in self.guessed_letters else "_" for char in self.word])

    def draw_trial(self):
        self.canvas.delete("all")
        self.canvas.create_oval(320, 100, 380, 160, outline="white", width=2)
        self.canvas.create_line(350, 160, 350, 280, fill="white", width=2)
       
        if self.mistakes >= 1:
            y_pos = 180 if self.mistakes < 4 else 140
            if self.boltgun_sprite:
                self.canvas.create_image(150, y_pos, image=self.boltgun_sprite)
            else:
                self.canvas.create_rectangle(50, y_pos-30, 250, y_pos+30, fill="#600", outline="#f00")
           
        if self.mistakes >= 4:
            self.canvas.create_line(230, 145, 320, 130, fill="red", width=2, dash=(5, 2))

    def make_guess(self):
        guess = self.entry.get().upper().strip()
        self.entry.delete(0, tk.END)
        if len(guess) != 1 or not guess.isalpha() or guess in self.guessed_letters: return

        self.guessed_letters.add(guess)
        if guess in self.word:
            self.play_sfx("correct")
            self.word_display.config(text=self.get_display_word())
            if "_" not in self.get_display_word(): self.end_game(True)
        else:
            self.mistakes += 1
            self.trigger_mistake_sequence()

    def trigger_mistake_sequence(self):
        self.status_label.config(text=STAGES[self.mistakes-1])
        self.narrative_display.config(text=random.choice(NARRATIVES))
        self.draw_trial()

        if self.mistakes < self.max_mistakes:
            s_map = {1: "load", 2: "load", 3: "bolt", 4: "aim", 5: "safety"}
            if self.mistakes in s_map: self.play_sfx(s_map[self.mistakes])
        else:
            self.play_sfx("eat_bolt")
            self.root.after(600, self.execute_fire)

    def execute_fire(self):
        self.play_sfx("fire")
        self.screen_shake(intensity=25)
        self.draw_explosion()
        self.root.after(600, lambda: self.play_sfx("eject"))
        self.root.after(1500, lambda: self.end_game(False))

    def end_game(self, won):
        if won:
            messagebox.showinfo("審判結束", f"真理已明。{VICTIM_NAME} 被判無罪釋放。")
        else:
            msg = (f"{VICTIM_NAME} 在一場由拼寫錯誤引發的危機中喪生。\n"
                   f"[ 已淨化 ]\n"
                   f"我們將永遠記住知識、進步與錯誤的代價。\n"
                   f"正確的單字是: {self.word}")
            messagebox.showerror("帝皇的裁決", msg)
        self.root.destroy()

if __name__ == "__main__":
    root = tk.Tk()
    game = BolterTrial(root)
    root.mainloop()